## Preprocessing Pipeline

**Raw Dataset**  
↓  
**Remove Duplicates**  
↓  
**Clean Text**  
↓  
**Train / Validation / Test Split**  
↓  
**Tokenization**  
↓  
**Convert Words → Numbers**  
↓  
**Padding / Truncation**  
↓  
**Prepare Labels**  
↓  
**Save Processed Data**

In [6]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [7]:
train_df = pd.read_csv("../data/raw/train.csv")
test_df = pd.read_csv("../data/raw/test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (16000, 2)
Test shape: (2000, 2)


In [8]:
train_df = train_df.drop_duplicates(subset="text").reset_index(drop=True)

print("Train shape after removing duplicates:", train_df.shape)

Train shape after removing duplicates: (15969, 2)


In [9]:
X = train_df["text"]
y = train_df["label"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

Why stratify=y?

Because we have 6 emotion classes.

It makes sure the class proportions remain approximately the same in:

training
validation

For example, if ~33% of the dataset is joy, we don't want the validation set to accidentally contain only ~20%.

In [10]:
X_test = test_df["text"]
y_test = test_df["label"]

X_train, y_train → model training

X_val, y_val     → tuning / early stopping

X_test, y_test   → final evaluation

In [11]:
def clean_text(text):
    text = text.lower()
    text = text.strip()
    return text

In [12]:
X_train = X_train.apply(clean_text)
X_val = X_val.apply(clean_text)
X_test = X_test.apply(clean_text)

In [13]:
MAX_VOCAB_SIZE = 10000

tokenizer = Tokenizer(
    num_words=MAX_VOCAB_SIZE,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

1000 words
↓
less memory
↓
but many useful words become OOV

50,000+ words
↓
larger model
↓
more parameters
↓
rare words may add little value


reasonable vocabulary
        ↓
manageable model size
        ↓
captures most common language

What does <OOV> do?

OOV = Out Of Vocabulary.

If the tokenizer encounters a word that isn't among the allowed vocabulary, it maps it to a special token:

"I am supercalifragilistic"
          ↓
       <OOV>

So the model doesn't completely fail when it encounters an unknown w

In [14]:
tokenizer.fit_on_texts(X_train)

In [22]:
tokenizer.word_index

{'<OOV>': 1,
 'i': 2,
 'feel': 3,
 'and': 4,
 'to': 5,
 'the': 6,
 'a': 7,
 'feeling': 8,
 'that': 9,
 'of': 10,
 'my': 11,
 'in': 12,
 'it': 13,
 'like': 14,
 'so': 15,
 'im': 16,
 'for': 17,
 'me': 18,
 'but': 19,
 'is': 20,
 'have': 21,
 'was': 22,
 'am': 23,
 'this': 24,
 'with': 25,
 'not': 26,
 'be': 27,
 'about': 28,
 'on': 29,
 'as': 30,
 'you': 31,
 'at': 32,
 'just': 33,
 'when': 34,
 'or': 35,
 'all': 36,
 'because': 37,
 'more': 38,
 'do': 39,
 'can': 40,
 'really': 41,
 'up': 42,
 'are': 43,
 't': 44,
 'very': 45,
 'by': 46,
 'been': 47,
 'know': 48,
 'myself': 49,
 'if': 50,
 'what': 51,
 'how': 52,
 'out': 53,
 'time': 54,
 'get': 55,
 'little': 56,
 'had': 57,
 'will': 58,
 'from': 59,
 'now': 60,
 'being': 61,
 'people': 62,
 'they': 63,
 'would': 64,
 'want': 65,
 'he': 66,
 'them': 67,
 'some': 68,
 'her': 69,
 'still': 70,
 'think': 71,
 'one': 72,
 'him': 73,
 'who': 74,
 'ive': 75,
 'even': 76,
 'an': 77,
 'life': 78,
 'its': 79,
 'we': 80,
 'make': 81,
 'there': 

only on training data.

We don't fit the tokenizer on validation or test data because that would allow information from those datasets to influence our preprocessing.

In [15]:
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [16]:
MAX_LENGTH = 50

In [17]:
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

In [18]:
print("X_train:", X_train_pad.shape)
print("X_val:", X_val_pad.shape)
print("X_test:", X_test_pad.shape)

X_train: (13573, 50)
X_val: (2396, 50)
X_test: (2000, 50)


In [19]:
y_train = y_train.to_numpy()
y_val = y_val.to_numpy()
y_test = y_test.to_numpy()

In [20]:
print(y_train[:10])
print(y_val[:10])
print(y_test[:10])

[3 0 0 1 4 0 1 4 1 1]
[4 0 1 1 4 1 3 1 1 0]
[0 0 0 1 0 4 3 1 1 3]


In [21]:
print("Vocabulary size:", len(tokenizer.word_index))

Vocabulary size: 14047
